# EPU E6 real-data GPU corrected study R2

Frozen before data access. This exploratory successor runs 20 repeats of five-fold stratified group cross-validation on the real Optical Digits dataset, with fixed same-bank controls, paired inference, Holm adjustment, and a shuffled-label control. It refuses CPU fallback and does not modify the retained E5-R1 ledger.


In [ ]:
from pathlib import Path
import base64, hashlib, io, json, tarfile
PAYLOAD_SHA256 = "4be88ea4a13ac1581edb3e03bfdd8ecc273b40e3f9cf23ba85b53198abf05526"
PAYLOAD_B64 = """H4sIAAAAAAAA/+09a3Pbxnb3M3/FFpk2oALCpGwpthJmqsh0ooljuZLs6wzLwYXIpYQIBGgAlES7mumP6C/sL+l57AK7IKjHTW5yO0NMYuGxj7Nnz573LuV8EcjdJ2+Pj06PDo5e+7/mafKX3/nqwrX77Bn9hav+9+tut6fv+X2vt727/RfR/b0BaboWeRFm0P0f0dc/4fW5JYRzJbM8ShNnTziDt+86x4P91y/3T/c7g93OD/i87XhYbJqln2QSLIpxMAkLicW3u9u7nV63s93jIvliDm3JiczXNmaXzLHjIJNhzv0f90Q6i4pCTkRxIUWaRedREsadjwuZLUWYjC/STBQym8H9REyiiUjSQoTzebwURTrvXIpMFlkkr8L4G3G8LcZplslxkYuztLgQZ3KaZhKqQlMT1Q0A9QSGE4pskfgMm7yKJjIZyyBLYxrm4MPb10fH+6dHx78EOKAARxQcHB0fDw5OBy+Dk3cHB4OTk6Pj4PDNq6Pjn+HV978EMPzgP/Z7wf6bl8H7XvB+h1vHznJZQLuIfHiBnRVRscSe8nF0GRWdWIZZIuI0nAQTwADAn86LaBzG4gLGfZ0h6IngT9QqQn0zh5HKCUB9jejvff3i6/qnqQyLRUazs/us/nEch3lO34ZdT/Q8se2Jp5545okdT+x64mtPPPfEi5GqF0djmeSEn+9PXnaedg7icJFLQcicRDlMw9kCMXwdAerNgX0jJjKLruATkNRMvDs4FEdqeMdynJ4nUQFUIdKp+NEY7UtrtPhWZkE0yYPwKozi8IymahrGuYQStzyRMLQsmgF6wzhYQLMIrLwJx0VHYUKcZ+liLsIpNCZCWAgh0Ns5ENEcvgOA6aLopNPONI0nYp7JSTRG0HIaU5RwbZ5WroLY2+7ycoE6+LjD9C4lPQ17XcRsD/552gWMPsPHnS7geRfvvsZ/nndfAJ7xa6+Lt70efu9t47feU/r3Gb3fwWZ6VLH3Nb15TvdUd7vbfUpz5SRplMsAGF0yCTOgKKDvkIZBU+3DZHf93g7+qyrMZHGRMrjOJMIFFMwWcRE5nnD0kgxguV3iC16U5eNlkjzFv3F6nslzRzWYTmRcUfw8S4u0WM5lHsAEMeEh4tTclksYGw0ujS9lxWDKkzmRyAyiBMgtGsPsZ8A1cpjdKMsLATRUZCF8xBmFRQEv4qUmoLyQc+z0uXoO4/lFCM9df1e9OZMFvtj1u+pFOfQKwDicnU2wVK8spfDRUAawrMoAkmhcT/VaImQFB1ZD6uUsvAmQ1omyut2SuKNkKjNkUxVex2kyZdZFvb3Y0SNJ0wLWYzgHqPJwNo9plfdQ2mrEhjDNkyCPzhOcktmiKCmEiuly03AWxctrpKcKYV3dDxHJHBiD4mY/pjEw6smvIGRxEYpwnKV5Dpw9k8CFx+NFFo6X4mI5T3HaZLm471iqJbeDCZ8BQWOxRXIto/MLXK8zGSa6m4ba+TfV0s4B0MJDwvBYmEigQ0BCh5aLSM9ymV0xFkSIUoO4A1SMkhyQLGQ4vlDsQ8mVakbKSTLGBouJQOe5Ir5POELJ+O708PXh6S9qcDyTMF85YcFYYAJIfZELc01WcKlh63EIHkcsr2ScVy3jYpktZoGcTqENnr9e+RUqR1oX4BLiuz4XAXF0DUzyxc6/CqYWUZIV3C0Ag98JYCU45x2ec+SfiewgtoB7VlQl5uJbIhuHur311uLl+Oj7dyenb0C2/n2oKRQSgLN1/5+i4M3RG1AqBseHR8ed10c/HA9+eCgqptENdApchJkj8JKM9a1Hk0yYAd+DTjoPQFTnwZhSJXHtPQxj4Tkwc+DrqBlSZcGQPRSVJ6f739+xzizJpnBoYRV5C0jRQqJ6M4/DsUSudidd3U9WPULAQwjL76kRwr9KsMtzwM2VDGgYqSFhQQMHVs4qurCFpBaJHdCZZGyiOAcOpvij6ArSebq4hj7JLGXS8ARKtEVYqnWW8NeiJM8DZPH5BbQQaDZP6NiuGCNoItdAcxXImfy4wKEj0AfvXu6jZq46KYAkYZLEc/FD9D3o+aDOwWzM0qyU5lNQAYHLBzQ9pJK+Pjr4CfTwN0fB/sufD09RTYcGKyAjUHjT2Rw01CBcgLqTRZ+ob1uDhHURzQKagjAjmTa4mcdpFhbQOeAKTBPWd3HJpHOSctp88MXpBavDoO6D4p4DDgXrrYJ1/mkENpAn8hRIOgLs52TNcJEO0NivaQTtkVTPZjRFHkzRBGYowfo4V1k6WYz5y8nR6X7ZuQe2k1gkMAughqB6L4DoQMUGeVvC57RuW3+2Abq5/tRLsv8HjNqXPw/82eQf0cc9/p/tp7tPa/6f7rPe043/54+4vhAD4LIVHxvsEttVbhPUkhfjMWgNwEyOt1utU2RTuZyHwABlvBTKc2SVyxaoMAv2FnloQQOflR2bb6n+WsDVsugGLbWfwvPzWEI1MA0YBGLOvjgsxCSVzBrBjoymS2RtObZFmgAYWiDQ5KQ12OkcgzQtCjmbo8SYnMuMeTCZjeM0Ftchgp9FYYzcXuvtxHyrAaTQNDRXVMzbb7WOt0Xl3ALGKm/kmLwb2CV5D+aZ7MDYRbiYRAVIzwXBB7LL9mm1HuXTQtdPqZ3VHFw+thzlLY0AEWJZlPsdJQ4NUQTwR2j4oYTKCW9kHon3O9AgSGiyW1p/+/Sl6Auy7LY+ia+E2+vQQ3vLhVs2ZNtbH+EL32+x6Hc/tdt/I+gBQ4DSWd4qbSGRApE02mIP8rq0TK/LiuXlI0lKUMoWk6VYgJUFJrJqJkf/0RSUI3beoC5FAneibDbSfDuAxmiiJCuZpS3WmnEBLOakd7I2DFI6BCo+C5NLZWPQeDXuSEqjCgYFp4s4poKtyzdvPFwKliLeMRRxJPKQlAlPK34MfYd6rSxKr0W2tGlg6x5Zgw5FfrGYToHqlV6ndUNdzBekVAGlhy1UvsgDylAozQtJwxf74gBWH2hB8Vk4vsTyExlHZ1Kt+ExOAc8ThXjttmPfHAxhfAl2BRDH+DJvUHV8MbiSCcLKiw0whewCiVg+WK3y7lWSgD+01qlJ/kbjMS8l/2kB+fPlP6SPu+X/bre7s1uX/7u7vY38/yMux3FekaAmPQCkf7X2UAgzXYgTkMlWiNCHeq0Wee+DYLpAph4EKGDSDPkiLFA2KFst/S47B0YHhpV6HudX+vb8UzTX92AkXQCz0Y/Ylb6HpX2h79Oc+57DOyiuO35rFAHzvEBJpJ8zYJJp+QS8soji8mmZ69simskS6GQxmy9RqiYlgPklRTC4e/XgKy0h13AYcRu7YAxiOswC8obrwq+VWDgupYJdZ4bifly2rc3pIAcRBbzwLIxBg5CVna0/THt8V2sNuw5yGUvihrrVk1I6/oDC8adXKHmtivNoLhF8UU7HpQz0y1pR4rfI31Golz1w9OEExIXMWq0vxLGy98UZhu4yEOmWnwIUj3NQlYqLGWp9oBiCkkQSTKmMiqXnfivNfZlcRRmQJczCRE5DkJKuc/Du+9f7J8Ffj45/Onm7fzAIDo7evDr8AWMTe8+6L3b3njvtctqx/1ar9X5wfHJ49AZ0oHWR0Nbx0dEpfEdac4H2oxgov+3D1KXxlXTbPpA5yBxoCwABOgu3d3aDs2Uhcxc9J3JP0ENbdL5DnWSP3BGgwS2yRFO/z7W4fNu/kDdAS2DHu227VezbxSWwR9A0NmkBgGUBUiBOfm7rBkmYBom8NprzhII3PfsViIVafwOqPjdPTfFQ/dklKEQuP+T902yBbqIbmMQgvaTHNlUhVw7VQwXbdW5gJkCeg0afnPedRTHtwITgasMQZ6z6wYsYzgTWImPEUwXYFZIU/W30oGRFcCmXunvQXtLrIAmT/it057TLtriqTwN2nf9MHI0CWrPaTnBptKh2KmSG1zDlLk39E+FYrNBpWyhl7Gh7o8/AY+O5C63w52halvDPJZCqDsC3xb/0habBNKuV0hFGKgV67koBjjjS550KfVmIPt33iLpBlqWZ6yySMvqtG3BKyGKYG7tZDl16YjhqU9v667CEaHRXb1gd1LMFKGJgvIE+m3PYVzka6YvqnhljvzYsDh564vNtCSS90lixg4WMnQcOf6VyQwfro37UVc+nmTAqrAsAUvGu33swbNosU8ZWrsBTS1sjSRGwUuKD8/miTr4wHlSZicn548Uk9CMjau626wAdw4yAHFQgrfGk7jE3BoDL8HtJ/GDb9c3uAC0UdR5LXGJgIxYRrJUul2d3bh9Wc+FSXZ/eBOzgLWeEi30rnost0etuP9vaevp3wm36jwWYSuk1O5YdDU8Jd17CXQGLH2dhsghRjMqJiykwAE9vpbJRKAB+1FgQTKnAknpBKfVyt2KdasY/l+N1GCr0R2Nfe1VwC6MCYKjCF8YlPhjfTNQyy3IUQoxC2CS9RhAVb6IxWQ3Bx7JQEKhiQeCpGAVTJUZygzFwaZkFiWLg+R4oVD5qA1m4JFKtHnlKuRjQBHwIc/qgqnpiguH/PnyYAlctdp8xhrgLWDV9XbmjbnwM27ghyKN+zxOXUs4n0Sw35FKC/gruK8Z1fu7jG1e3CMJkfV1cWQBisnS5lW+BH8hOb3tlSZlrHHXADtirmJBRYPxZjEllBlokYNg5ZS/2cnxPGF4TvTQFJWp5Qk5lAjKRkGu+2HsQsmCQxngbUMVNKmxdgXYA7K6GtLVtAMrOUhCx3EYD6h6Bu4IG9dtQOFmgPyUE/Ye9Uu6He8lzFs7noLTsEYsd0ipCZaQYAWo+36o+c1BC4Xk4ouepAjNKTJr+0EDO1ehBn4EGoBaUx1Ku8+30OcjBcTpfKr0GWKWpdyj8YkVEBnSmQS0/G/APoRyCjBJfvaqa4QH48FomE9esYSHVGAzXMEYEGMHlyWieAqFWaT82kj2xtB/LpKA9RCuAuM2UDFMlh2bB6l6pIB8SKF1jOh8MpYx69wS5yHKaHs+cInadAd5QoQTmDu1DYx8X0l0iruMItXBjiuboe8J2PiRDmKw+NzAqv1PIuK+KGZyoWyGaU5VY/CE2s/NZeOPWmJHqp0MNao7UbletgMaRS+xqSO1VEFxfgJVAU8xF2iBCK/RaZIE5e2hLKmYIIsm1vhMHmPtQaHzpDtdByDdD1M1vRiWohF16h9hlUKqv3ko/iuda79vN0A5VawB1p2cV4Q+ahm0El9VNLFYk4subAitZo2kaRFWZaUpXHGJ39Ko9Ao2lRDmXf7tKpcZKMil1hT9Yy++tZ65AhmDdCqxy55RLnWeXohDW8qutltVv3M/qe9SB99AE5UfWUbRE4qeqYMzs02vdu7SZtzKtfiylT5gHzP7dGiJpQG2NBS6skOcpmPr8R/GFhiZXZ+D+dgArDS3Zc6IEZ5qcNzYRYeYrKcMf/TGUkq5lkCCDsfMxqwXMeSlQsXyDerZK6wDRSq8BEsOu1q0qUfH57tTOW5tZrIjmKdgul0l6nRgxESaIz/jn1jFli20ZoW1NJDG8w9gaWViwQV1hH+aFgxaq+XWm2agCjuJdABFNtQaLcx2NUpgbWi9E+aJGGTIpFWdXRer2plF6dQKR2wSUkhMm59I1muHc1ZEpiagJ9PfhbHNb/y7m/qlV4Auxr4JRzMWAL4ZxsSRnns6XpEYEGmi2P863WtI1S4JnGOfou5rLYXekKZ5h8umhTvPAF9WyipJp6ppF2z6M0OoRYMNolu6Qwo5qwB0NDuibfUJsqX/GwO5lXii3EEt1VkaBOQL/kU2dBCUiua/zsLiQmeqNMuO5nC2QOAe1qpWn0wIlDZHKlt20gs9uQBEH6eWqicXMVc2CHpIDZ5OfpLuNiJsPucFRY1OamAz7YFXEEqFvcdmVrxj79buAWo7/QjlXvagvHvymGddXq2t7qxqXDWO7mVM1kDmrZ7Tc15L4gwnSbOdBZLlaoSLOKjgtSPcb6s5Jx7BANoiUJ4x1NDOT8C7Lisfb0Zi2Zr10CGlg/DE6gXyKX7goCo1O6t8q5Cs9ARP4SxVhVTvg/EH7VYNC0Cj9G62pf4BIv7m/SRrFI5pc/lbZXlLuR6Dam7up9ub3Z6PAFu/nnU8fyzev0sLgk0niTxcJUWAY+6C+BBdp4S5hUcxhIEBxeotRv9dtE3NbJWJrAVHzmtRtwlXEilmpy7o1SamqeyycAXgpJ2RHNlJfNFVJz6BVdA3Pw4p522Sr+2iKKx0tS87ZbuJIo6/iUAG8dxGEutk8jiNL6W9qH/lpwgZWGLu0U0el4X5gCmnjph3crOV3NUaqvR7jqHQKoeE6pFZhIso9IISVOoJYE1e44yqMFgLzMT65h2FkfAF6Y155Ss7CYnyBJv9OtzTLMUZfVJpQ1xiDxxUMTSiPPqHoQ/OVPhmFgdqpKcNbwsSMfhZANOBAnsssxx7QZGYXkEdN9l3813xvmd84CG1p0uehanpkOiBVlUkWXisre5wm6HxK4H+XW7EIhRVMKPdxESYFBvyoLk379k677d1V4sXXO2WIz0jzDubNdGFmgtdJA5dvHKs1JWg7BFELPT+ePmAisD2qxxttSOvhsVBdxlr7UZREMSaAzaKjHu+ZWkdI5qAfQktWuvwKOWHynSYmsDQwcGCgY9jpqcU6ehBR0Xi+Kl1DyDBd7mGLsd22aAv3E2hk2m4CVzUloNAT4VpjwJeKRi7SeBbwLgx3rkmEfJxo0ytmUMaWzJeMsTSbKLeycp+pRvyokLMclY1LuewrvRDf7dG/w17pU2QPYrZIaM8eelOJ0tlpoFyVqpd25VcNE1D4XWhbxazbOMMSBAUlrZUVDO5edoBSRj15NM00Q+4M9VxoFmUrN1n3jGrnqapsYpsLKJxy+gbuyXU54KmWE8Zq9cqyxBXvvWQ3CuhaC4zH1x0txlUpfTX1y1D5zC/ivyiKj0sbWM8kV/3WA94rAScG3tlTYVsjEoQjwX0T8McMQuGg4DXLq+o9Dw8+8I1ZwyF8uDjHNO624Rd0jG0kiktY+S5cwzPQYdVeyZIpm1mTP3NPezPcOBVMe2UzOttmtZ6nd1v1uZZjNwTrN7B2E+3R/KBeYpkGUc7vyZtR8nwVrqnKWY3LOJznuKuTpxqaVnd2jO5MJuOLWZhdUux4rd5eUUdYanzIjxPX3e4+e44bjp89bzfqvmePrWDGYJcAXZYmwCeVpkU8l63kaCZ9TDkOiMGBja51Cjm+BHYJRRBpJacwvCnPDXZgFA9BOT+7HwSF2WYQtFiQE+aKpnWL0ett/B+GvrX11FpsDfOl3nh6vytu0y9g9mnnstHwkxKiJ6Ine9s2m3D0AEty1S9wo393RPwZZK0miFzGU7BxMNZhT7yR11LLl2FroCFsOxx2fRR5MNvw/1Mf4y1P1RO+7dKbrnrDf/FN9URlRqWPleM2Zg84BvLKaDliem/nZZVaBErrJvzdTszhEABp2Jiz4YJJCDoL5tsoJ78OBeFXs/966HIfrJ0Mp0iFL8vWOSsbE/AxVb8MVarE+P5qVBIGfKUiLxpy/gtG4qhdgs+l4R1Cph46vRHCjnw1l4XLb6toFg3i2X2QlwBxqyg9LcjLzZt9S4347IR6fy1wYb4FK9MZ83bAKqMHndCubmTooL/3276oXpzVX4yd0UqkuA50ff+5BTE70j0RwH8658gKj2g8N5CTJ+rnIpjs0nXAVHUaolqNV5VOxYlOiojZVc2h9Iq/ezY7WRPTqRbjOujtQxuagF+Fi5ouJ4zHXy0SDa96AwRneZ3KclY93m2jx2PllqlOSy/gfXNdbVkyEqeaV1nJcNGptqCjat7un5wEJ4PXr4LTwckpxj40JAGnUKLwNNIu16QC1vhudbBONS79bqS5LWiQbroo5ovCyON8MM/lmnfnYBo5kDXZif+ohvSGXNRpzVQyI5RXSvGSUjj3yG4AAyFKoXBQQ25SL1RGaJl2yqNYRWhFhetrlLOmqxjiCywbrPYhAJPYIx+ekZvtMh0EH4KlnZ9CxQ32TyfpiF0WAXdmBmBxumUlo9e9O5tF7w3WJwAJymxBz2SR2yT7wZZ5akCrdjWI/11lL9Vs8eVqXJhmFkAIOL8Y7TYzXZjRYOaeGPkmYDcuq2+R9e0+WUYNV90jBsDUc8xzi8CssI4qop3Y6nSiGoJX2KwzDhNQ1sZhHNDQqxVsDBY5eA0uw/6oZpc/tZu6UfnuyBlU5ruRBne7ll4xofxkUFKrwsB6+h68eX94fPTm58Gbso5hl82XYEYRh1EbHnx+oyFBq9vR36xi6sY1LQZyrjpkL5opffeM1bs3HZAaN7b861uPkw2DqyiPzmKpki1Jua2S+impFpNNg/eHJ4ffvx4ELwfvDw8GJ5rf3qplrrZKKLcWstBR5VdkF2CgxAQSXAmZq5yqbJK2afi4ic91G1Kd2VmzRNcbWtWYjtPs/sQLjQ52L0eJIQPWH8jEOSYMiF1Hn8k0UrabMaZaDOmfc2BW1WqA7uMOmGobowdxElR2dkAOOjI/mP8BjwlwuwG8qBbT8f5faRHF/vknThugbQm4B4m3Jeha0O9N0bBDAZAjr3HPS592D+B+BaiyZ41OuUjIe2q7oipMcVL9qBbBH18BtE2bcdwkoPOK8n7VBKf7w7ypnZ9K7LOPlE/C6FfeURNA8tUIjoEFlAoAOiAwRb6vuc/GVz51jREKECKaIe7V+SFKzsq+GBqNj9ri36xPZVf18evrPhvEPr0NGNJldZhNSMNz2isNfwgIJhgF32DangUmfLGem1og0bGkv1y/GgtVrx5XaluaeKm4r7NP69CujofPByMnprEXazXQj5e9/cpdY5us7kVzD/rkEdWHkPXxCDIMCMbA4ftOfDY9x60hqzS32sG6AfhTpK17R/t4jtNMWqqCJCUY5X1t19XU+ax24Nw++cwrGW6QpOAPVb51fOIKoDlbu7OGe73dEZ7Htwo8Xrzdv69DloqUFIeuoGqubOyCN13yRv5rlXtcv0zxsKa28grf0QhYCsAFG+IB66v8NkbfPHsMyvjyDuff3eirm9PNpv+dljtNpHfnctYiVzs311nTTdf6Udzpkrx72PmQARrRcPXbtRUtkjGqmu/vmB8iFaNas5MU53FtI2uE+1c6lXG100cRhYkZPrIS4VxNffFWpIae0uZ2HzVFGlEVAI9ElD1NVTOl0/vPxKo+AJSMbmL0FmbvQYlR+7chxWhoLVru1bnXMyMrA6xhva2tqIyVcodFPU7oKS3NUuG16mEFmB7mZrwDRwpUr75y2+s5i21NDWuGxmiox2DoQ5jla2Krr8bS1m6EJjujftXCYhWTUWenVTuU74HcbOWh4Df2vJag4nSMeXrncXoGMq7cNlAp1lXjTVtL6hfgOc0mlnnXdK0Eau0ArUAvCzyYUN1NQLVIrnBIwilfCc/gkIY6usfh7RirA0/904Fu3ve5snB0o/cvztt7+q2FV8uOORTWOKm688ctrd+m8NwxjGb9Ci/00PFm9nKXPPo1kVTu3xUvvhJqC7yGvsqhoLMg7VWu0zisHQS46NDLGMe4fqN8igmqUsVO6vTcaNZNHTqHKDm3DttWfSZghhBknwE0vYdgPaRWyOGPBNdywZjwUoPsSwxKj+iKe7GlvOwqgBfoY0ndWE4LlQuSYR64uucDPOQkoEVZT+Sycz9wtyX0qaa8851Z1sh0ShcZefibfEr1ZARKQrCJo/Jy13Y+6gljUx2mykJGDeGpToubxmGRpAmePOpqtKmApr0oEUGBGXg2E+qGPKqSv2Nh4O57dOpzPsJTO9DJ73bUBrXSvqzht8YIaCYe3CmV/j16tXZkMvMyRq8S5jBxyQBPp9FZm2/MJCV8xd2J/iMceXyCrWUOGmQ1ssSUfcT1Xp3E6wfI17gmA2c6qesnQz+2xSGec7/SYMM5y/c2rHTLtUBWBw6vNnXPOfrcor2WOUqlk3fw3zJpCbk8Kli148QbjtFeP9x25Ta15g/a3qvlj1qEqWiBMuJMQ7c6Dn00XH/A/Ah/9KDbBVkU2QIQE0T7fMyz2tOMG7/WAc88qasPz8fL9lZHlH1oa2AW+kzX8ppsBsJNGXfFYy2NxOVyc6hXnmgf5evcv8a58KOVLd9V9SEeX13xUXVudL+WAGZOgLHS6UxpTyyAYWQYhzUTwx80Z00/GzDCX6bg6SJfsSlcFX4/30uEtb2EeGI04bo6F9tEQe1Y7ZFxYPZ3eEA1nn6o00IoK5OPA6+wFj+AftZA5K4BiU9Et0FpKrBGcVsHc9sAOpe/H5buKlh2oklaixjb0OCD1EntcsoDzvnGM39h4sUO/tKBSXujWlYErSKTnaCxgu8o0gc9YMiSlh1+0WiqPtqt2TkcDuVwMaaIIziv9g9fIw0y/1V0X4KuVkDZpA4lfiFe0T7Jew7VFGl5YDqqKmyzq1sW63jPWyeVy3klBDTs8hQ8IPaz88BYz5rADuoq8qZw0ZG/LqyjU+lpeGVSrM3lzFP5y4TK7a6ZN6/QBvXskIq5Ywav5pz+nS6zGKOjtq/adHXbxrZ9zGxqiKDUgjsNFSm/am0CWC26Q92sCOq1Dl4zkd/Gpq3D1RKPl7VOMXPdYOr3efDqPwlga2Kqr4hC7HWw6mnJK7nSWt7UK1pJx+XJ/1bW08ovFZBK0PxDASNTw6lXHOoFjzpmteRXi9lDoOTB1UIVsCOTXbSqyLCRiPHq6PXL4OfB6fHhwYk/zq+ctnGSXRkVfuipdrDG4wmpD9plo101nva5eGW6PL4iHVGPx2vKaW9yIFSp6l5Ttrm3miVunB3CB/j28bRO/yVQ3F/phavP36Mh4AFTeZ9H065VZdfEhQwn6LT9xnqLtpDyeqp6+WIGYnNpU2yVQqeicmZ+e+132ipys78Y/N349TV1Z3zTAhDTT9StmRuz+mMb9VdG6VKR0r87tkbHoq+jxoraYliQ20qnJlnms7XymlkDmh6NH+7cKGBkB1bp7OYRYfXfxaiGV/tkream/KYff3l7dPrj4GRwUibwMSGolG9YVbRMqs6/aD6slk1aDEc7jncXQPi9WofOS5WDxnHcPfG3z0ae2O3ffKswdmR4sRUu98TnZiTf+goc54svxNtMdvK5HPMx5CW1VfAYhkQhZyhbdaGKbxA+SgnidMTW1mfaRfVlNPlydCv+97//R6gXzCnh5daW1nf0J376crTn705vvxFrTtCYOviLNNbZ7R31i1flz9PgnsnsKozXtzFUfZo64peo9lDnnmj+3OPPI98x6eAr5JcKnW9qyphjT6zu/sRW4dQvhCm95vOKTPjSEhwKQ3tNBUv08hzz5BmEfTw4eff69MSfTUBSMOUXqH6hz9XHU8tdGlO7QWBQU7MwiaaoC2DGcV7F2UkFw8Sm6pwqlQWcoVvfdbactu31pFNXo5xTl9t8UD++Qt6NKavOz/tvDl+VKbT108JMMNDXow+SjRkhRar6b7dHVQ5peTqtPZj7uLup0B8c/fz29eB0QJIx4h+ts6ExOBJ73dC/7+jTzGs52/aRvWhk2MnGD8rwNrz0DvvZc4v7lel0Ykvl06mbVRfR38+v/wTh8jsyfJvYvHJKreR8JQP0LkrQ492VI4iznLQTfba4v5+dL1CreUtf9Fm89OCHE1CU1HfX6XQwSbyDmrbDp7mkSR/oBhTwoADLyrmzLg8F901geuTbksahDHlRuRb9wXp5dYoTPvllfrpxjlyGM2CEcIwc9nocZ8WXazWu0Ky2Cxo9MFCSM+n0GLCc/t0tsUhijLYYuMHP+GsP+mTUbGmekKD8/Lh/wei6vX5Qen/vw89tVrvtbsZyXogB/aHfr8PDisYWizMHT96V6tmnjRBWJIjYqPqNFEw9v5sPoU0weEm6MqIvwGlHHQmD1gBH2w8C5KSUCA2yOg/P8TMySvx629Srzkw1wX7CPb07HvDKsOqp2JVZfc247GVnVvB0dYOCMMQFiwya14MgN1kQ4JILAkefeInrb/PTGZtrc22uzbW5Ntfm2lyba3Ntrs21uTbX5tpcm2tzba7Ntbk21+baXJtrc22uzbW5Ntfm2lyba3P9U13/ByvI8Y8AoAAA"""
payload = base64.b64decode(PAYLOAD_B64)
assert hashlib.sha256(payload).hexdigest() == PAYLOAD_SHA256
root = Path("/kaggle/working/epu-e6-source")
root.mkdir(parents=True, exist_ok=True)
with tarfile.open(fileobj=io.BytesIO(payload), mode="r:gz") as archive:
    archive.extractall(root, filter="data")
print(json.dumps({"status": "PASS_EXACT_SOURCE", "payload_sha256": PAYLOAD_SHA256, "bytes": len(payload)}, indent=2))


In [ ]:
import json, os, subprocess, sys
source = root / "epu_e6"
output = Path("/kaggle/working/epu-e6-realdata-gpu-r2")
print("Free-compute policy: paid compute is not authorized.")
subprocess.run([sys.executable, str(source / "study.py"), "--self-test"], cwd=source, check=True)
subprocess.run([sys.executable, str(source / "study.py"), "--output", str(output)], cwd=source, check=True)


In [ ]:
import json, shutil
output = Path("/kaggle/working/epu-e6-realdata-gpu-r2")
manifest = json.loads((output / "MANIFEST.json").read_text())
hypotheses = json.loads((output / "HYPOTHESES.json").read_text())
archive = shutil.make_archive("/kaggle/working/epu-e6-realdata-gpu-r2-evidence", "gztar", output)
print(json.dumps({
    "status": manifest["status"],
    "version": manifest["version"],
    "records": manifest["records"],
    "gpu_prediction_batches": manifest["gpu_prediction_batches"],
    "outcomes": hypotheses["outcomes"],
    "negative_control": hypotheses["negative_control"],
    "claim_boundary": manifest["claim_boundary"],
    "downloadable_archive": archive,
}, indent=2, sort_keys=True))
